# Phase 8 — Codeflow Simulation Validation

The ▶ Play data-movement engine. Given the **ordered walkthrough** a codemap already produced (real tree-sitter symbols + real call edges), it generates a coherent, *illustrative* execution trace:

```
step[i]:  INPUT  →  TRANSFORMATION  →  OUTPUT      (output[i] ≈ input[i+1])
```

**Repository-agnostic:** the structure is real (only symbols that exist in the map are simulated); the LLM only fills in plausible *data* grounded in each function's real signature / docstring / source. Degrades to a mechanical, signature-grounded trace with no LLM key.

One **batched** LLM call over the whole chain keeps the data coherent across boundaries and costs one request, not N. Needs the DB stack up (`docker compose up -d`) and, for real values, an LLM key (else the honest mechanical fallback).

In [ ]:
import os, sys, json
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.analysis.codemap import build_codemap
from archaeologist.analysis.simulation import simulate_flow
from archaeologist.rag.llm import has_api_key, active_model
print("LLM:", active_model(), "| key set:", has_api_key())

## 1. Build a walkthrough, then simulate the data flowing through it

Change `QUESTION` to anything about the ingested repo. The codemap picks the real symbols + order; the simulator fills in the data.

In [ ]:
QUESTION = "How does Flask dispatch a request to a view function?"

cm = build_codemap(QUESTION)
node_ids = [n["id"] for n in cm["nodes"]]
print(f"codemap: {len(node_ids)} nodes  (curated={cm['curated']})")
for n in cm["nodes"]:
    print(f"  step {n['step']}  {n['qualified_name']}  ({n['file']}:{n['line']})")

In [ ]:
sim = simulate_flow(node_ids, QUESTION)
print(f"source={sim['source']}  simulated={sim['simulated']}  truncated={sim['truncated']}")
print(f"scenario: {sim['scenario']}\n")

id2name = {n["id"]: n["qualified_name"] for n in cm["nodes"]}
for st in sim["steps"]:
    name = id2name.get(st["node_id"], st["node_id"])
    print(f"▼ {name}   [{st['confidence']}]")
    print(f"    IN : {st['input']['summary']}   {json.dumps(st['input']['fields'], default=str)[:120]}")
    print(f"    DO : {st['transformation']}")
    print(f"    OUT: {st['output']['summary']}   {json.dumps(st['output']['fields'], default=str)[:120]}")
    if st.get("branch_taken"):
        print(f"    BRANCH: {st['branch_taken']}")
    print()

## 2. Checks

- **one step per node, in order** — the trace maps 1:1 onto the real walkthrough,
- **grounded** — every `node_id` is a real symbol from the codemap,
- **honest** — without an LLM key, `source == "mechanical"` and every step is `representative`.

In [ ]:
assert len(sim["steps"]) == min(len(node_ids), 14), "one step per node (capped at 14)"
assert [s["node_id"] for s in sim["steps"]] == node_ids[:len(sim["steps"])], "order preserved"
assert all(s["node_id"] in id2name for s in sim["steps"]), "every step is a real symbol"
if not has_api_key():
    assert sim["source"] == "mechanical"
    assert all(s["confidence"] == "representative" for s in sim["steps"]), "honest fallback"
print("OK — grounded, ordered, and honest about its confidence.")

## 3. Repository-agnostic — try a totally different flow

Same engine, different question → different real symbols, different data. Nothing about RAG/pipelines/embeddings is baked in; the trace adapts to whatever the codemap surfaces.

In [ ]:
for q in ["How is the application context pushed and popped?",
          "How does url_for build a URL for an endpoint?"]:
    cm2 = build_codemap(q)
    s2 = simulate_flow([n["id"] for n in cm2["nodes"]], q)
    print(f"\nQ: {q}\n   scenario: {s2['scenario']}")
    for st in s2["steps"][:5]:
        nm = {n['id']: n['qualified_name'] for n in cm2['nodes']}.get(st['node_id'], st['node_id'])
        print(f"     {nm:38.38}  {st['input']['summary']:22.22} -> {st['output']['summary']}")